In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("processed_complaints.csv")

# -----------------------------
# Normalize features to 0-1
# -----------------------------

df["severity_norm"] = df["severity"] / 10
df["traffic_norm"] = df["traffic_impact"] / 10
df["public_norm"] = df["public_impact"] / 10
df["weather_norm"] = df["weather_risk"] / 10
df["frequency_norm"] = df["complaint_frequency"] / 10

# Cap days_pending at 30 days
# This prevents extreme values like 1494 days
# from dominating the score.
days_capped = df["days_pending"].clip(upper=30)
df["days_norm"] = days_capped / 30

# safety_risk is already 0-1
df["safety_norm"] = df["safety_risk"]

# -----------------------------
# Create priority score
# -----------------------------

df["priority_score"] = (
    0.25 * df["severity_norm"]
    + 0.20 * df["traffic_norm"]
    + 0.20 * df["public_norm"]
    + 0.10 * df["weather_norm"]
    + 0.10 * df["days_norm"]
    + 0.10 * df["frequency_norm"]
    + 0.05 * df["safety_norm"]
)

# Keep score strictly between 0 and 1
df["priority_score"] = df["priority_score"].clip(0, 1)

# Save processed training dataset
df.to_csv("training_data.csv", index=False)

print("Training data created!")
print("Shape:", df.shape)

print("\nPriority score statistics:")
print(df["priority_score"].describe())

print("\nSample priority scores:")
print(df[["complaint_type", "safety_risk", "priority_score"]].head(10))

Training data created!
Shape: (5000, 24)

Priority score statistics:
count    5000.000000
mean        0.360140
std         0.098109
min         0.181424
25%         0.283723
50%         0.357167
75%         0.415708
max         0.771045
Name: priority_score, dtype: float64

Sample priority scores:
               complaint_type  safety_risk  priority_score
0     noise - street/sidewalk       0.2133        0.376089
1  homeless person assistance       0.1689        0.269933
2     noise - street/sidewalk       0.2133        0.375755
3     noise - street/sidewalk       0.2133        0.375905
4     noise - street/sidewalk       0.2133        0.375793
5         noise - residential       0.1283        0.181599
6            street condition       0.5798        0.621020
7               lot condition       0.1689        0.368445
8            street condition       0.5798        0.625423
9                 taxi report       0.1689        0.368445
